In [ ]:
!pip install -q langchain langchain-community langchain-huggingface \
    langchain-text-splitters langchain-chroma pypdf chromadb \
    sentence-transformers transformers accelerate

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "MapReduce.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Loaded {len(pages)} pages")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading free embedding model (downloads ~90MB first time)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./rag_db",
)

print(f"Done! Indexed {len(chunks)} chunks.")

In [ ]:
from transformers import pipeline

print("Loading explanation model...")

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=300,
    do_sample=False,
    device_map="auto",
)
print("Model ready!")

def ask(question):
    relevant_chunks = vector_store.similarity_search(question, k=3)
    context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

    prompt = f"""<|system|>
You are a helpful assistant. Using only the context below, give a clear and
detailed explanation. Do not make up anything outside the context.</s>
<|user|>
Context:
{context}

Question: {question}</s>
<|assistant|>
"""

    result = generator(prompt)
    full_text = result[0]["generated_text"]
    answer = full_text.split("<|assistant|>")[-1].strip()
    return answer

print(ask("Explain the MapReduce"))

In [ ]:
print(ask("Explain about GitHub"))